# 12 · Streamlit demo UI (live agent view)

**Purpose.** Launch the Streamlit app (`app/streamlit_app.py`) from Colab and open it through a temporary public URL. The app shows which agent is running, the current step, a progress percentage and a live feedback log, then the final detections with the reason for every accept / reject.

| | |
|---|---|
| **Proposal objective** | Demo of Objective 1 (the three-agent pipeline) |
| **Inputs** | uploaded image or the COCO sample; optional tuned thresholds from `results/tuning/selected.json` |
| **Outputs** | interactive UI only (nothing is written to `results/`) |
| **Runtime (T4)** | ≈3 min to start (models load once) |

> Run cells top to bottom. Every expensive stage is **resumable**: if Colab disconnects, reconnect,
> re-run from the top and it continues where it stopped (results are cached on Drive under `results/`).

In [ ]:
# ===================== Bootstrap (identical in every notebook) =====================
import os, sys, subprocess, shutil  # paths, sys.path, shell commands (git clone, pip), folder copy
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = (not IN_COLAB) and os.path.isdir("/kaggle/working")    # Kaggle notebook (needs Internet ON, see docs/KAGGLE_GUIDE.md)

# Where the project lives on Google Drive. Upload the whole repo folder there (see docs/COLAB_GUIDE.md),
# or set REPO_URL and it will be cloned into DRIVE_PROJECT on first run.
DRIVE_PROJECT = "/content/drive/MyDrive/detect-verify-decide"
REPO_URL = ""  # e.g. "https://github.com/<you>/detect-verify-decide.git"
BRANCH = "kaggle-version"  # git branch to take the latest code from (change it if you push to another branch)

if IN_COLAB:
    from google.colab import drive  # Colab only: mount Google Drive
    drive.mount("/content/drive")
    if not os.path.isdir(DRIVE_PROJECT) and REPO_URL:
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, DRIVE_PROJECT], check=True)
    elif os.path.isdir(os.path.join(DRIVE_PROJECT, ".git")):   # pick up the latest pushed code (never touches your notebooks/results)
        subprocess.run(["git", "-C", DRIVE_PROJECT, "fetch", "origin", BRANCH], check=False)
        subprocess.run(["git", "-C", DRIVE_PROJECT, "checkout", "FETCH_HEAD", "--", "src", "configs", "requirements.txt", "tools", "app"], check=False)
    PROJECT_ROOT = DRIVE_PROJECT
elif IN_KAGGLE:
    PROJECT_ROOT = "/kaggle/working/detect-verify-decide"
    if not os.path.isdir(os.path.join(PROJECT_ROOT, "src", "dvd")):
        import glob
        attached = glob.glob("/kaggle/input/*/src/dvd") + glob.glob("/kaggle/input/*/*/src/dvd")   # project attached as a Kaggle Dataset
        if attached:
            shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), PROJECT_ROOT)
        elif REPO_URL:
            url = REPO_URL
            try:   # private repo: add a Kaggle secret named GITHUB_TOKEN (Add-ons -> Secrets)
                from kaggle_secrets import UserSecretsClient
                url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
            except Exception:
                pass
            r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT_ROOT], capture_output=True)
            if r.returncode:   # (the message deliberately does not echo the command, which could contain the token)
                raise RuntimeError("git clone failed: is Internet ON, REPO_URL / BRANCH right, and for a private repo the GITHUB_TOKEN secret set?")
            subprocess.run(["git", "-C", PROJECT_ROOT, "remote", "set-url", "origin", REPO_URL])   # never keep the token in .git/config
        else:
            raise RuntimeError("Set REPO_URL (and BRANCH) above, or attach the project as a Kaggle Dataset.")
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

assert os.path.isdir(os.path.join(PROJECT_ROOT, "src", "dvd")), f"src/dvd not found under {PROJECT_ROOT}"
os.environ["DVD_ROOT"] = PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
# Install dependencies (Colab / Kaggle already have torch/torchvision with CUDA). ~1-2 min per fresh runtime.
if IN_COLAB or IN_KAGGLE:
    %pip install -q -r "{PROJECT_ROOT}/requirements.txt"

> **Heads-up.** Colab cannot expose a port directly, so this notebook starts a **Cloudflare quick tunnel**.
> It downloads the `cloudflared` binary from Cloudflare's GitHub releases and prints a temporary
> `https://<random>.trycloudflare.com` URL. **Anyone who has that URL can use the app on your GPU session**
> until you run the last cell (or the runtime stops). Do not share the URL.

In [ ]:
# 1. cloudflared (tunnel client)
if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(["wget", "-q", "-O", "/usr/local/bin/cloudflared",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)
print("cloudflared ready")

In [ ]:
# 2. start the Streamlit app in the background (log: /content/streamlit.log)
import time, re  # stopwatch and regular expressions (find the tunnel URL)
os.environ["DVD_ROOT"] = PROJECT_ROOT
app_proc = subprocess.Popen(
    ["streamlit", "run", f"{PROJECT_ROOT}/app/streamlit_app.py", "--server.port", "8501", "--server.headless", "true",
     "--browser.gatherUsageStats", "false"],
    stdout=open("/content/streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)
print("streamlit running:", app_proc.poll() is None)
if app_proc.poll() is not None:
    print(open("/content/streamlit.log").read()[-2000:])

In [ ]:
# 3. open the tunnel and print the URL
tunnel_log = open("/content/cloudflared.log", "w")
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501"], stdout=tunnel_log, stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
    if m:
        url = m.group(0); break
print("Open this URL in a new tab:" if url else "No URL yet - see /content/cloudflared.log", url or "")

## Using the app
1. Wait for **Loading models** to finish (first start downloads the weights).
2. Upload an image, or press **Use a COCO sample image**.
3. Adjust the uncertain band (τ_low, τ_high) and the crop margin if you like, then press **Run pipeline**.
4. Watch the five agent cards, the progress percentage and the log update live; then open the tabs
   (**Result**, **Verifier crops**, **All candidates**, **Run log**).

If the page shows errors, check `/content/streamlit.log`.

In [ ]:
# 4. stop everything (also closes the public URL)
for p in (globals().get("tunnel_proc"), globals().get("app_proc")):
    if p is not None and p.poll() is None:
        p.terminate()
print("stopped")